# 07 — Proporção de `CPF HighDelivery` e `CPF HighRead`

**Objetivo:** dar suporte à **Parte 3 — Desenho de Experimento (Teste A/B)**, fornecendo estatísticas de baseline a nível de CPF que podem ser utilizadas como métricas primárias/secundárias e para o cálculo de tamanho de amostra.

## Convenção de classificação

- **HighDelivery**: taxa de entrega (`status ∈ {delivered, read}`) ≥ **90%**.
- **HighRead**: taxa de leitura (`status == read`) ≥ **75%**.

Aplicáveis em nível de telefone (Telefone HighDelivery / Telefone HighRead) e
de CPF (CPF HighDelivery / CPF HighRead). Este notebook calcula:

1. **Proporção de `CPF HighRead`** — taxa de leitura = `read / total_disparos` por CPF, com limiar **≥ 75%**.
2. **Proporção de `CPF HighDelivery`** — taxa de entrega = `(delivered + read) / total_disparos` por CPF, com limiar **≥ 90%**.

**Fonte:** `outputs/processed/base_disparo.parquet` (gerado em `01_preprocessing.ipynb`).

In [1]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
PARQUET_PATH = PROJECT_ROOT / 'outputs' / 'processed' / 'base_disparo.parquet'

df = pd.read_parquet(PARQUET_PATH)
print(f'Linhas: {len(df):,} | CPFs únicos: {df["cpf"].nunique():,}')
df['status_disparo'].value_counts()

Linhas: 392,917 | CPFs únicos: 268,436


status_disparo
read          271563
delivered      85404
failed         27257
sent            5533
processing      3160
Name: count, dtype: int64

## Agregação por CPF

- `total_disparos`: número total de disparos do CPF.
- `n_read`: disparos com `status_disparo == 'read'`.
- `n_delivered_or_read`: disparos com status `delivered` ou `read` (entrega efetiva — `read` implica que a mensagem foi entregue antes de ser lida).
- `taxa_leitura = n_read / total_disparos`
- `taxa_entrega = n_delivered_or_read / total_disparos`

In [2]:
df['is_read'] = (df['status_disparo'] == 'read').astype(int)
df['is_delivered_or_read'] = df['status_disparo'].isin(['delivered', 'read']).astype(int)

agg_cpf = (
    df.groupby('cpf')
      .agg(total_disparos=('status_disparo', 'size'),
           n_read=('is_read', 'sum'),
           n_delivered_or_read=('is_delivered_or_read', 'sum'))
      .reset_index()
)
agg_cpf['taxa_leitura'] = agg_cpf['n_read'] / agg_cpf['total_disparos']
agg_cpf['taxa_entrega'] = agg_cpf['n_delivered_or_read'] / agg_cpf['total_disparos']
agg_cpf.head()

,cpf,total_disparos,n_read,n_delivered_or_read,taxa_leitura,taxa_entrega
0,10000012749921281024,2,0,2,0.000000,1.0
1,10000045642210965504,2,2,2,1.000000,1.0
2,10000073440807188480,3,2,3,0.666667,1.0
3,10000164307433218048,1,1,1,1.000000,1.0
4,10000199941913628672,3,3,3,1.000000,1.0


In [3]:
agg_cpf[['total_disparos', 'taxa_leitura', 'taxa_entrega']].describe()

,total_disparos,taxa_leitura,taxa_entrega
count,268436.000000,268436.000000,268436.000000
mean,1.347588,0.714557,0.916277
std,5.217498,0.442875,0.266455
min,1.000000,0.000000,0.000000
25%,1.000000,0.000000,1.000000
50%,1.000000,1.000000,1.000000
75%,1.000000,1.000000,1.000000
max,2670.000000,1.000000,1.000000


## Proporção de `CPF HighRead` (≥ 75%) e `CPF HighDelivery` (≥ 90%)

In [4]:
LIMIAR_HIGH_READ = 0.75
LIMIAR_HIGH_DELIVERY = 0.90
n_cpfs = len(agg_cpf)

n_high_read = (agg_cpf['taxa_leitura'] >= LIMIAR_HIGH_READ).sum()
n_high_delivery = (agg_cpf['taxa_entrega'] >= LIMIAR_HIGH_DELIVERY).sum()

prop_high_read = n_high_read / n_cpfs
prop_high_delivery = n_high_delivery / n_cpfs

resultado = pd.DataFrame({
    'metrica': ['CPF HighRead (taxa_leitura >= 75%)',
                'CPF HighDelivery (taxa_entrega [delivered+read] >= 90%)'],
    'n_cpfs_atende': [n_high_read, n_high_delivery],
    'n_cpfs_total': [n_cpfs, n_cpfs],
    'proporcao': [prop_high_read, prop_high_delivery],
})
resultado['proporcao_pct'] = (resultado['proporcao'] * 100).round(2).astype(str) + '%'
resultado

,metrica,n_cpfs_atende,n_cpfs_total,proporcao,proporcao_pct
0,taxa_leitura >= 90%,187256,268436,0.697582,69.76%
1,taxa_entrega (delivered+read) >= 90%,242592,268436,0.903724,90.37%


In [5]:
print(f'Total de CPFs analisados: {n_cpfs:,}')
print(f'CPF HighRead     (taxa de leitura >= 75%): {n_high_read:,} ({prop_high_read:.2%})')
print(f'CPF HighDelivery (taxa de entrega >= 90%): {n_high_delivery:,} ({prop_high_delivery:.2%})')

Total de CPFs analisados: 268,436
CPFs com taxa de leitura >= 90%:  187,256 (69.76%)
CPFs com taxa de entrega >= 90%:  242,592 (90.37%)


## Uso na Parte 3 (Teste A/B)

- **Baseline da métrica primária** (CPF HighRead) e da **métrica secundária** (CPF HighDelivery) para dimensionar o experimento.
- A proporção de CPF HighRead/HighDelivery serve como referência para o efeito mínimo detectável e para definir o público elegível ao teste.